In [1]:
import pandas as pd #required for dealing with dataframes
import numpy as np  #required for data processing
import matplotlib.pyplot as plt #required for data visualization
import seaborn as sns #required for data visualization
from itertools import combinations  # Create combinations of pairwise features
from itertools import product  # Create combinations of numerical and categorical features
import math


In [3]:
df = pd.read_excel('My Superstore (2).xlsx')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16798 entries, 0 to 16797
Data columns (total 30 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Days To Ship         16798 non-null  int64  
 1   Category             16798 non-null  object 
 2   City                 16798 non-null  object 
 3   Container            16798 non-null  object 
 4   Country / Region     16798 non-null  object 
 5   Customer Name        16798 non-null  object 
 6   Customer Segment     16798 non-null  object 
 7   Customer_ID          16798 non-null  int64  
 8   Department           16798 non-null  object 
 9   Discount             16798 non-null  float64
 10  Item                 16798 non-null  object 
 11  Number of Records    16798 non-null  int64  
 12  Order Date           16798 non-null  object 
 13  Order Priority       16798 non-null  object 
 14  Order Quantity       16798 non-null  int64  
 15  Delivered Qty        16798 non-null 

In [11]:
print(f" My data has {df.shape[0]} rows and {df.shape[1]} columns")

 My data has 16798 rows and 30 coulmns


In [15]:
df.dtypes

Days To Ship             int64
Category                object
City                    object
Container               object
Country / Region        object
Customer Name           object
Customer Segment        object
Customer_ID              int64
Department              object
Discount               float64
Item                    object
Number of Records        int64
Order Date              object
Order Priority          object
Order Quantity           int64
Delivered Qty            int64
Order                    int64
Postal Code            float64
Product Base Margin    float64
Profit                   int64
Region                  object
Row                      int64
Sales                    int64
Due Date                object
Ship Date               object
Ship Mode               object
Shipping Cost            int64
State                   object
SubRegion               object
Unit Price               int64
dtype: object

In [19]:
df.head()

,Days To Ship,Category,City,Container,Country / Region,Customer Name,Customer Segment,Customer_ID,Department,Discount,...,Region,Row,Sales,Due Date,Ship Date,Ship Mode,Shipping Cost,State,SubRegion,Unit Price
0,1,Storage & Organization,Suva,Small Box,Fiji,Joy Corbett,Small Business,1656,Office Supplies,0.10,...,AsiaPac,1,173,2010-02-01 00:00:00,2010-02-01 00:00:00,Regular Air,5,Central,NaN,6
1,2,Storage & Organization,Bowie,Large Box,United States of America,Anita Hahn,Home Office,2211,Office Supplies,0.08,...,North America,2,177,2010-03-01 00:00:00,2010-03-01 00:00:00,Express Air,35,Maryland,East,96
2,2,Binders and Binder Accessories,Los Angeles,Small Box,United States of America,Ernest Oh,Consumer,949,Office Supplies,0.06,...,North America,3,116,2010-04-01 00:00:00,2010-04-01 00:00:00,Regular Air,3,California,West,41
3,0,Chairs & Chairmats,New York City,Jumbo Drum,United States of America,Scott Bunn,Corporate,68,Furniture,0.00,...,North America,4,1168,2010-02-01 00:00:00,2010-02-01 00:00:00,Delivery Truck,49,New York,East,292
4,2,Chairs & Chairmats,New York City,Jumbo Drum,United States of America,Scott Bunn,Corporate,68,Furniture,0.07,...,North America,5,4039,2010-04-01 00:00:00,2010-04-01 00:00:00,Delivery Truck,45,New York,East,101


when we inspect columns types we found that date columns are documented as objects so we will change type to date


In [26]:
df["Due Date"] = pd.to_datetime(df["Due Date"]).dt.date

In [30]:
df["Order Date"] = pd.to_datetime(df["Order Date"]).dt.date

In [32]:
df["Ship Date"] = pd.to_datetime(df["Ship Date"]).dt.date

In [34]:
df.duplicated().sum().sum()

0

No duplicated rows

In [39]:
df.duplicated(subset = ['Row']).sum()

0

🎄Detection of Inaccuracies and Inconsistencies🎄


In [43]:
Numerical_Features = df.select_dtypes(include = ['float', 'int']).columns.tolist() #you can use include = np.number
Categorical_Features = df.select_dtypes(include = ['object', 'category', 'bool']).columns.tolist()
print('Numerical Features are:', Numerical_Features)
print('Categorical Features are:', Categorical_Features)

Numerical Features are: ['Days To Ship', 'Customer_ID', 'Discount', 'Number of Records', 'Order Quantity', 'Delivered Qty', 'Order', 'Postal Code', 'Product Base Margin', 'Profit', 'Row', 'Sales', 'Shipping Cost', 'Unit Price']
Categorical Features are: ['Category', 'City', 'Container', 'Country / Region', 'Customer Name', 'Customer Segment', 'Department', 'Item', 'Order Date', 'Order Priority', 'Region', 'Due Date', 'Ship Date', 'Ship Mode', 'State', 'SubRegion']


In [50]:
df.isnull().sum()


Days To Ship              0
Category                  0
City                      0
Container                 0
Country / Region          0
Customer Name             0
Customer Segment          0
Customer_ID               0
Department                0
Discount                  0
Item                      0
Number of Records         0
Order Date                0
Order Priority            0
Order Quantity            0
Delivered Qty             0
Order                     0
Postal Code            6985
Product Base Margin       0
Profit                    0
Region                    0
Row                       0
Sales                     0
Due Date                  0
Ship Date                 0
Ship Mode                 0
Shipping Cost             0
State                     0
SubRegion              7316
Unit Price                0
dtype: int64

In [52]:
df.isnull().sum().sum()


14301

In [54]:
pd.pivot_table(df,index = ["Region"], values= ["SubRegion", "Postal Code"], aggfunc = "count")

,Postal Code,SubRegion
Region,,
AsiaPac,94,0
EMEA,293,0
Latam,0,0
North America,9426,9482


we found that North american only has  both subregions and postal code , while Latam hasnt have postal code or subregion

In [57]:
df.describe()

,Days To Ship,Customer_ID,Discount,Number of Records,Order Quantity,Delivered Qty,Order,Postal Code,Product Base Margin,Profit,Row,Sales,Shipping Cost,Unit Price
count,16798.000000,16798.000000,16798.000000,16798.0,16798.000000,16798.000000,16798.000000,9813.000000,16798.000000,16798.000000,16798.000000,16798.000000,16798.000000,16798.000000
mean,2.033218,1754.262829,0.056633,1.0,26.219848,25.595547,59334.677223,52312.177316,0.500638,882.161388,8399.500000,1790.082093,12.857007,89.330158
std,2.301389,978.716372,0.085028,0.0,27.184655,27.190245,31827.183249,29561.428445,0.153102,2355.553751,4849.309246,4645.264802,17.253488,290.342578
min,0.000000,1.000000,0.000000,1.0,1.000000,0.000000,3.000000,1001.000000,0.036000,-4301.000000,1.000000,1.000000,0.000000,1.000000
25%,1.000000,912.000000,0.020000,1.0,8.000000,7.000000,29857.750000,28352.000000,0.380000,29.000000,4200.250000,90.000000,3.000000,6.000000
50%,2.000000,1778.000000,0.050000,1.0,16.000000,15.000000,72895.500000,53081.000000,0.510000,134.000000,8399.500000,336.000000,6.000000,21.000000
75%,2.000000,2593.000000,0.080000,1.0,38.000000,37.000000,88699.000000,77530.000000,0.590000,656.000000,12598.750000,1391.000000,14.000000,86.000000
max,92.000000,3403.000000,0.950000,1.0,288.000000,288.000000,91591.000000,99362.000000,0.850000,60251.000000,16798.000000,99130.000000,165.000000,6783.000000


In [71]:
df[df["Days To Ship"] > 30]

,Days To Ship,Category,City,Container,Country / Region,Customer Name,Customer Segment,Customer_ID,Department,Discount,...,Region,Row,Sales,Due Date,Ship Date,Ship Mode,Shipping Cost,State,SubRegion,Unit Price
8130,84,Envelopes,Sao Paulo,Small Box,Brazil,Robyn Horowitz,Corporate,249,Office Supplies,0.05,...,Latam,7246,773,2012-03-22,2012-03-22,Regular Air,5,São Paulo,NaN,11
8519,92,Pens & Art Supplies,Porto Alegre,Wrap Bag,Brazil,Vicki Thomas,Home Office,963,Office Supplies,0.02,...,Latam,7259,76,2012-03-31,2012-03-31,Regular Air,1,Rio Grande do Sul,NaN,2
9583,84,Envelopes,Richfield,Small Box,United States of America,Brenda Nelson Blanchard,Corporate,250,Office Supplies,0.05,...,North America,7254,199,2012-03-22,2012-03-22,Regular Air,5,Minnesota,Central,11
11644,92,Pens & Art Supplies,Redwood City,Wrap Bag,United States of America,Virginia Rivera,Home Office,964,Office Supplies,0.02,...,North America,7266,20,2012-03-31,2012-03-31,Regular Air,1,California,West,2
15775,31,Telephones and Communication,Rotterdam,Small Box,United States of America,Milton Harrell,Corporate,3252,Technology,0.08,...,North America,7220,541,2012-01-26,2012-01-26,Regular Air,4,New York,East,196
15776,31,Telephones and Communication,New York City,Small Box,United States of America,Peter Brooks,Corporate,3251,Technology,0.08,...,North America,7208,2344,2012-01-26,2012-01-26,Regular Air,4,New York,East,196


In [69]:
df[df["Discount"] > 0.94]

,Days To Ship,Category,City,Container,Country / Region,Customer Name,Customer Segment,Customer_ID,Department,Discount,...,Region,Row,Sales,Due Date,Ship Date,Ship Mode,Shipping Cost,State,SubRegion,Unit Price
210,2,Tables,Woodmere,Jumbo Box,United States of America,Alex Watkins,Small Business,1603,Furniture,0.95,...,North America,211,9,2010-01-18,2010-01-18,Delivery Truck,29,New York,East,179
261,2,Tables,Cleveland Heights,Jumbo Box,United States of America,Carlos Hess,Small Business,263,Furniture,0.95,...,North America,262,14,2010-01-23,2010-01-23,Delivery Truck,46,Ohio,East,32
358,1,Tables,Wheeling,Jumbo Box,United States of America,Marsha P Joyner,Corporate,911,Furniture,0.95,...,North America,359,110,2010-01-02,2010-01-02,Delivery Truck,70,West Virginia,East,219
369,1,Tables,Cuyahoga Falls,Large Box,United States of America,Denise Carver,Corporate,397,Furniture,0.95,...,North America,370,62,2010-03-02,2010-03-02,Regular Air,69,Ohio,East,154
709,1,Tables,West Islip,Large Box,United States of America,Alison Peters Wooten,Corporate,605,Furniture,0.95,...,North America,710,77,2010-03-15,2010-03-15,Express Air,69,New York,East,154
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3985,4,Tables,Derry,Large Box,United States of America,Howard Elliott,Home Office,943,Furniture,0.95,...,North America,6862,178,2011-01-29,2011-01-29,Regular Air,69,New Hampshire,East,209
4018,2,Tables,Glen Cove,Large Box,United States of America,Molly Browning,Home Office,1625,Furniture,0.95,...,North America,2190,115,2010-08-18,2010-08-18,Regular Air,69,New York,East,209
4053,1,Tables,Cumberland,Large Box,United States of America,Kerry Beach,Home Office,2352,Furniture,0.95,...,North America,3370,67,2010-01-24,2010-01-24,Regular Air,69,Maryland,East,71
4054,2,Tables,Port Chester,Large Box,United States of America,Jane Clayton,Home Office,3188,Furniture,0.95,...,North America,14293,63,2013-07-18,2013-07-18,Regular Air,69,New York,East,209


In [79]:
df["Ship Date"] = pd.to_datetime(df["Ship Date"])
df["Due Date"] = pd.to_datetime(df["Due Date"])

In [81]:
df["Days Difference"] = (df["Ship Date"] - df["Due Date"]).dt.days

In [83]:
df["Days Difference"].describe()

count    16798.000000
mean         0.496428
std          1.279281
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          5.000000
Name: Days Difference, dtype: float64

to check for delay we create new column called day difference and found that mean delay is zero days and maximum number of delay was 5

In [86]:
l =df[df["Days Difference"] > 0]

,Days To Ship,Category,City,Container,Country / Region,Customer Name,Customer Segment,Customer_ID,Department,Discount,...,Row,Sales,Due Date,Ship Date,Ship Mode,Shipping Cost,State,SubRegion,Unit Price,Days Difference
1751,1,Chairs & Chairmats,Upper Saint Clair,Jumbo Drum,United States of America,Sara Sykes Davies,Consumer,1902,Furniture,0.05,...,5974,1535,2011-10-08,2011-10-09,Delivery Truck,45,Pennsylvania,East,101,1
1752,2,Chairs & Chairmats,Perry Hall,Jumbo Drum,United States of America,Gayle Christian,Consumer,305,Furniture,0.10,...,7346,4058,2012-07-31,2012-08-01,Delivery Truck,26,Maryland,East,501,1
1759,0,Tables,Bossier City,Jumbo Box,United States of America,Joe George,Consumer,2464,Furniture,0.08,...,1276,3356,2010-11-04,2010-11-05,Delivery Truck,32,Louisiana,South,228,1
1760,1,Tables,Fayetteville,Jumbo Box,United States of America,Sherry Hurley,Consumer,1574,Furniture,0.10,...,8643,403,2012-05-04,2012-05-05,Delivery Truck,46,North Carolina,South,32,1
1761,1,Chairs & Chairmats,Palm Coast,Jumbo Drum,United States of America,Jon Ayers,Consumer,1531,Furniture,0.00,...,10587,3751,2012-02-10,2012-02-11,Delivery Truck,30,Florida,South,121,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12296,2,Binders and Binder Accessories,Plainfield,Small Box,United States of America,Sheryl Doyle Block,Small Business,2316,Office Supplies,0.10,...,15665,416,2013-11-07,2013-11-10,Regular Air,1,Connecticut,East,21,3
12302,2,Labels,Central Islip,Small Box,United States of America,Eugene Kerr,Small Business,1026,Office Supplies,0.07,...,3140,61,2010-01-09,2010-01-12,Regular Air,1,New York,East,3,3
12307,2,Paper,Mansfield,Small Box,United States of America,Jan Ellis,Small Business,1993,Office Supplies,0.02,...,9613,201,2012-02-05,2012-02-08,Regular Air,20,Ohio,East,41,3
12317,2,Paper,West Milford,Small Box,United States of America,Jack Durham,Small Business,441,Office Supplies,0.10,...,5062,54,2011-12-03,2011-12-06,Regular Air,6,New Jersey,East,5,3


In [92]:
l = df[df["Days Difference"] > 0].shape[0]
l

2821

In [104]:
####the percentage of delayed order
percent_delay = l * 100/(df.shape[0])
round(percent_delay,2)

16.79

We found that 16.79 % of total orders was delivered late

In [142]:
status_list = []

for x in df["Days Difference"]:
    if  x > 0:
        status_list.append("Late")
    else:
        status_list.append("On Time")

df["Status"] = status_list

We add a colum for delay df["Status"]

In [138]:
df["deliver_difference"] = (  df["Order Quantity"]- df["Delivered Qty"])

In [140]:
df

,Days To Ship,Category,City,Container,Country / Region,Customer Name,Customer Segment,Customer_ID,Department,Discount,...,Ship Date,Ship Mode,Shipping Cost,State,SubRegion,Unit Price,Days Difference,Status,deliver_state,deliver_difference
0,1,Storage & Organization,Suva,Small Box,Fiji,Joy Corbett,Small Business,1656,Office Supplies,0.10,...,2010-02-01,Regular Air,5,Central,NaN,6,0,On Time,0,0
1,2,Storage & Organization,Bowie,Large Box,United States of America,Anita Hahn,Home Office,2211,Office Supplies,0.08,...,2010-03-01,Express Air,35,Maryland,East,96,0,On Time,0,0
2,2,Binders and Binder Accessories,Los Angeles,Small Box,United States of America,Ernest Oh,Consumer,949,Office Supplies,0.06,...,2010-04-01,Regular Air,3,California,West,41,0,On Time,0,0
3,0,Chairs & Chairmats,New York City,Jumbo Drum,United States of America,Scott Bunn,Corporate,68,Furniture,0.00,...,2010-02-01,Delivery Truck,49,New York,East,292,0,On Time,1,1
4,2,Chairs & Chairmats,New York City,Jumbo Drum,United States of America,Scott Bunn,Corporate,68,Furniture,0.07,...,2010-04-01,Delivery Truck,45,New York,East,101,0,On Time,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16793,1,Computer Peripherals,Bend,Small Pack,United States of America,Randy Lucas,Small Business,3032,Technology,0.02,...,2013-01-29,Regular Air,2,Oregon,West,30,0,On Time,2,2
16794,1,Computer Peripherals,Loveland,Small Pack,United States of America,Sara O'Connor,Small Business,290,Technology,0.04,...,2010-09-26,Regular Air,5,Colorado,West,5,0,On Time,2,2
16795,3,Computer Peripherals,Caldwell,Small Pack,United States of America,Megan York,Small Business,2741,Technology,0.10,...,2012-08-24,Regular Air,2,Idaho,West,8,0,On Time,2,2
16796,2,Telephones and Communication,Apple Valley,Small Pack,United States of America,Janet Zhang,Small Business,2530,Technology,0.01,...,2011-05-19,Regular Air,1,California,West,56,0,On Time,2,2


In [144]:
df["QTY_status"] = (df["Delivered Qty"] == df["Order Quantity"]).map(
    {True: "Quantity Reached", False: "Not Reached"}
)

In [156]:
m = (df["QTY_status"] == "Quantity Reached").sum()
m

9237

In [168]:
percent_of_quantity_reached = m * 100/ df.shape[0]

In [172]:
print(round(percent_of_quantity_reached,2),"%")

54.99 %


In [174]:
df["Achieved"] = (
    (df["QTY_status"] == "Quantity Reached") &
    (df["Status"] == "On Time")
).map({True: "achieved", False: "not-achieved"})


In [184]:
service_level = (df["Achieved"] == "achieved").sum() *100 / df.shape[0]

In [186]:
service_level

44.60054768424813

In [190]:
print(round(service_level,2),"%")

44.6 %
